# Position probe inspection

Single-image inference and visualization for the position-between-objects probe.
- Hardcoded parameters (image, backbone, probe, environment, labels)
- Loads backbone, loads saved head state_dict, runs a forward pass
- Shows prediction vs ground truth, with optional attention overlay for non-linear probes

In [ ]:
from pathlib import Path
repo_dir = "/home/kargin/Projects/repositories/midvision-probe"
results_dir = "/shared/results/common/kargin/unreal_engine/probe"
# ---- User parameters ----
ENVIRONMENT = "winter_town_2"
IMAGE_PATH = Path(f"/shared/results/common/kargin/unreal_engine/dataset/position_between_objects/{ENVIRONMENT}/mid-objects/img_0000.jpg")
MODEL_NAME = "vggt_l16"  # backbone key
PROBE_NAME = "cls_efficient"  # options: cls_linear, cls_abmilp, cls_efficient
PERSPECTIVE = "camera"  # camera or human
REFERENCE_LABEL = "Snowman"
TARGET_LABEL = "Husky"
HEAD_ROOT = Path(f"{results_dir}/result/position_between_objects")
DEVICE_PREFERENCE = "cuda"  # "cuda" or "cpu"
IMAGE_MEAN = "imagenet"  # or "clip"
IMAGE_SIZE = 224
AMBIGUITY_DEGREES = 10 # 10 or 15
FRONT_DEGREES = 45
BACK_DEGREES = 135


In [ ]:
import torch
import numpy as np
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from typing import Optional
import sys
sys.path.append(repo_dir)

from evals.datasets.unreal_position import (
    LABEL_TO_INDEX,
    INDEX_TO_LABEL,
    _matching_json_for_image,
    _load_positions,
    classify_relative_direction,
)
from evals.models.vggt import VGGT1B
from evals.models.probes import ClassificationHead, EfficientProbing, ABMILPHead

device = torch.device(
    DEVICE_PREFERENCE if DEVICE_PREFERENCE == "cpu" or torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

# Set backbone parameters
if PROBE_NAME == "cls_linear":
    return_cls = False
    mean_pool = True
    efficient_probe = False
elif PROBE_NAME == "cls_abmilp" or PROBE_NAME == "cls_efficient":
    return_cls = False
    mean_pool = False
    efficient_probe = True


In [ ]:
def get_mean_std(image_mean: str):
    if isinstance(image_mean, (list, tuple)):
        mean = [float(m) for m in image_mean]
        std = [1.0, 1.0, 1.0]
    elif image_mean == "imagenet":
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
    elif image_mean == "clip":
        mean = [0.48145466, 0.4578275, 0.40821073]
        std = [0.26862954, 0.26130258, 0.27577711]
    else:
        mean = [0.0, 0.0, 0.0]
        std = [1.0, 1.0, 1.0]
    return mean, std


def load_image_and_label(
    image_path: Path,
    perspective: str,
    reference_label: str,
    target_label: str,
    image_size: int,
    image_mean: str,
    ambiguity_degrees: int,
    front_degrees: int,
    back_degrees: int,
):
    mean, std = get_mean_std(image_mean)
    transform = T.Compose(
        [
            T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ]
    )

    with Image.open(image_path).convert("RGB") as im:
        pil_img = im.copy()
    tensor_img = transform(pil_img)

    json_path = _matching_json_for_image(image_path)
    label_idx: Optional[int] = None
    label_name: Optional[str] = None
    if json_path is not None:
        positions = _load_positions(
            json_path,
            reference_label,
            target_label,
            human_label="Human",
        )
        if positions is not None:
            observer_key = "camera" if perspective.lower() == "camera" else "human"
            observer = positions.get(observer_key)
            if observer is not None:
                ref = positions["reference"]
                tgt = positions["target"]
                label_name = classify_relative_direction(
                    observer,
                    ref,
                    tgt,
                    ambiguity_degrees,
                    front_degrees,
                    back_degrees,
                )
                label_idx = LABEL_TO_INDEX.get(label_name)

    return tensor_img, label_idx, label_name, mean, std


def load_backbone(model_name: str):
    if model_name == "vggt_l16":
        model = VGGT1B(
            repo_dir="/home/kargin/Projects/repositories/vggt",
            add_norm=True,
            return_cls=return_cls,
            mean_pool=mean_pool,
            efficient_probe=efficient_probe,
        )
    else:
        raise NotImplementedError(f"Backbone {model_name} is not wired up in this notebook.")
    model.eval().to(device)
    return model


def build_head(probe_name: str, feat_dim: int, num_classes: int = 4):
    probe_name = probe_name.lower()
    if probe_name == "cls_linear":
        head = ClassificationHead(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    elif probe_name == "cls_efficient":
        head = EfficientProbing(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    elif probe_name == "cls_abmilp":
        head = ABMILPHead(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    else:
        raise ValueError(f"Unsupported probe: {probe_name}")
    head.eval().to(device)
    return head


def find_head_checkpoint(root: Path, model_name: str, probe_name: str, environment: str, perspective: str, reference_label: str, target_label: str) -> Path:
    if not root.exists():
        raise FileNotFoundError(f"Head root {root} does not exist")
    tokens = [
        model_name,
        probe_name,
        environment,
        perspective,
        reference_label,
        target_label,
    ]
    candidates = []
    for pt in root.rglob("*.pt"):
        path_str = pt.as_posix().lower().replace(" ", "-")
        if all(tok.lower().replace(" ", "-") in path_str for tok in tokens):
            candidates.append(pt)
    if not candidates:
        raise FileNotFoundError(f"No head checkpoint found in {root} matching tokens {tokens}")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def denorm_image(tensor_img: torch.Tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    img = (tensor_img.cpu() * std + mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def overlay_attention(image_np: np.ndarray, attention_map: Optional[torch.Tensor], head, backbone):
    if attention_map is None:
        return image_np
    attn = attention_map.detach().cpu()
    if attn.ndim == 3:
        attn = attn.squeeze(0)
    try:
        attn = attn.reshape(head.num_queries, image_np.shape[0] // backbone.patch_size, image_np.shape[1] // backbone.patch_size)
    except Exception:
        attn = attn[:, 1:]
        attn = attn.reshape(head.num_queries, image_np.shape[0] // backbone.patch_size, image_np.shape[1] // backbone.patch_size)
    attn = attn.mean(dim=0, keepdim=True)
    # attn = attn.max(dim=0, keepdim=True)[0]
    # attn = attn.min(dim=0, keepdim=True)[0]
    # attn = attn.std(dim=0, keepdim=True)
    # attn = attn[0:1, :, :]
    # attn = attn[1:2, :, :]
    # attn = attn[2:3, :, :]
    # attn = attn[3:4, :, :]

    # # attn has shape (1, H, W)
    # attn_flat = attn.view(-1)
    # # Find indices of the TOP-3 values
    # top3_values, top3_indices = torch.topk(attn_flat, 80)
    # print(top3_values)
    # # Set them to zero
    # attn_flat[top3_indices[78:79]] = 0.035
    # attn_flat[top3_indices[50:60]] = 0.02
    # # Reshape back
    # attn = attn_flat.view_as(attn)

    attn = attn.unsqueeze(0)
    attn = torch.nn.functional.interpolate(attn, scale_factor=(backbone.patch_size, backbone.patch_size), mode="nearest")[0].permute(1, 2, 0).numpy()
    attn = cv2.blur(attn, (8, 8))
    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    heatmap = plt.get_cmap("jet")(attn)
    heatmap = heatmap[:, :, :3]
    alpha = 0.25
    blended = (1 - alpha) * image_np + alpha * heatmap
    return blended


In [ ]:
# Load image and GT label (if metadata is present)
image_tensor, gt_idx, gt_label, mean, std = load_image_and_label(
    IMAGE_PATH,
    PERSPECTIVE,
    REFERENCE_LABEL,
    TARGET_LABEL,
    IMAGE_SIZE,
    IMAGE_MEAN,
    AMBIGUITY_DEGREES,
    FRONT_DEGREES,
    BACK_DEGREES,
)
print(f"Image loaded. GT: {gt_label if gt_label is not None else 'unknown'}")

# Load backbone
backbone = load_backbone(MODEL_NAME)
feat = backbone(image_tensor.unsqueeze(0).to(device))
# if isinstance(feat, (list, tuple)):
#     feat = torch.cat(feat, dim=-1)
# if feat.dim() > 2:
#     feat = feat.view(feat.size(0), -1)
feat_dim = feat.shape[-1]
print(f"Feature shape after flatten: {feat.shape}")

# Load head and weights
head_ckpt = find_head_checkpoint(
    HEAD_ROOT,
    MODEL_NAME,
    PROBE_NAME,
    ENVIRONMENT,
    PERSPECTIVE,
    REFERENCE_LABEL,
    TARGET_LABEL,
)
head = build_head(PROBE_NAME, feat_dim, num_classes=4)
state = torch.load(head_ckpt, map_location=device)
head.load_state_dict(state)
head.eval().to(device)
print(f"Loaded head from {head_ckpt}")

# Inference
with torch.no_grad():
    logits = head(feat)
    probs = torch.softmax(logits, dim=1)
    pred_idx = int(probs.argmax(dim=1).item())
    pred_label = INDEX_TO_LABEL.get(pred_idx, str(pred_idx))
    attn_map = head.attention_map if hasattr(head, "attention_map") else None

print(f"Prediction: {pred_label} (p={probs[0, pred_idx]:.3f})")


In [ ]:
# Visualization
image_np = denorm_image(image_tensor, mean, std)
probe_type = PROBE_NAME.lower()
if probe_type in ("cls_linear",):
    blended = image_np
else:
    blended = overlay_attention(image_np, attn_map, head, backbone)

title = f"GT: {gt_label if gt_label is not None else 'unknown'} | Pred: {pred_label}"
plt.figure(figsize=(4, 4))
plt.imshow(blended)
plt.axis("off")
plt.title(title)
plt.savefig("blended.png", bbox_inches="tight", pad_inches=0.1)
plt.show()

SAVE_IMAGE = True
file_name = "blended.png"
if SAVE_IMAGE:
    from PIL import Image
    blended_save = (blended * 255).astype(np.uint8)
    blended_save = cv2.resize(blended_save, (448, 448))
    im = Image.fromarray(blended_save)
    im.save(file_name)


In [ ]:
# Define IMAGES in the dataset from img_0000 to img_0033
IMAGES = [Path(f"/shared/results/common/kargin/unreal_engine/dataset/position_between_objects/{ENVIRONMENT}/mid-objects/img_{i:04d}.jpg") for i in range(15, 20, 1)]
for image_path in IMAGES:
    # Load image and GT label (if metadata is present)
    image_tensor, gt_idx, gt_label, mean, std = load_image_and_label(
        image_path,
        PERSPECTIVE,
        REFERENCE_LABEL,
        TARGET_LABEL,
        IMAGE_SIZE,
        IMAGE_MEAN,
        AMBIGUITY_DEGREES,
        FRONT_DEGREES,
        BACK_DEGREES,
    )
    print(f"Image loaded. GT: {gt_label if gt_label is not None else 'unknown'}")

    # Load backbone
    backbone = load_backbone(MODEL_NAME)
    feat = backbone(image_tensor.unsqueeze(0).to(device))
    # if isinstance(feat, (list, tuple)):
    #     feat = torch.cat(feat, dim=-1)
    # if feat.dim() > 2:
    #     feat = feat.view(feat.size(0), -1)
    feat_dim = feat.shape[-1]
    print(f"Feature shape after flatten: {feat.shape}")

    # Load head and weights
    head_ckpt = find_head_checkpoint(
        HEAD_ROOT,
        MODEL_NAME,
        PROBE_NAME,
        ENVIRONMENT,
        PERSPECTIVE,
        REFERENCE_LABEL,
        TARGET_LABEL,
    )
    head = build_head(PROBE_NAME, feat_dim, num_classes=4)
    state = torch.load(head_ckpt, map_location=device)
    head.load_state_dict(state)
    head.eval().to(device)
    print(f"Loaded head from {head_ckpt}")

    # Inference
    with torch.no_grad():
        logits = head(feat)
        probs = torch.softmax(logits, dim=1)
        pred_idx = int(probs.argmax(dim=1).item())
        pred_label = INDEX_TO_LABEL.get(pred_idx, str(pred_idx))
        attn_map = head.attention_map if hasattr(head, "attention_map") else None

    print(f"Prediction: {pred_label} (p={probs[0, pred_idx]:.3f})")

    # Visualization
    image_np = denorm_image(image_tensor, mean, std)
    probe_type = PROBE_NAME.lower()
    if probe_type in ("cls_linear",):
        blended = image_np
    else:
        blended = overlay_attention(image_np, attn_map, head, backbone)

    title = f"GT: {gt_label if gt_label is not None else 'unknown'} | Pred: {pred_label}"
    plt.figure(figsize=(4, 4))
    plt.imshow(blended)
    plt.axis("off")
    plt.title(title)
    plt.show()
